# Bedrock Module · Day 15 — AgentCore: Memory & Evaluations

**Phase 1 · Module 2: AWS Bedrock & AgentCore** · ~2.5 hrs

Day 14 gave your agent a governed home (Runtime, Gateway, Policy). Two things still separate a demo from a product:

1. **Memory** — a hosted agent should remember a customer *across sessions*, not just within one conversation. AgentCore **Memory** stores facts long-term and **streams** the relevant ones back into each new invocation.
2. **Evaluations** — before you ship, you need numbers: does the agent answer correctly, cite sources, stay in policy? AgentCore **Evaluations** run a suite of test cases against defined **metrics** and score the agent.

Today you build a long-term memory store the agent reads and writes, watch it change behaviour across two separate sessions, then define an eval suite and read its scores.

> **Keyless & offline.** `Fake*` classes named after real AgentCore components, same call shapes. Maps to real APIs in the closing table.

## 0. Setup — a tiny agent we can evaluate

We need something with *behaviour* to remember and score. Reuse a Day-14-style agent: it answers balance questions and refuses investment advice (a Barclays policy). Deterministic, so evals are repeatable.

In [1]:
import re, time, asyncio

class Settings:
    agent_id = "loan-assistant"
    region   = "eu-west-2"
cfg = Settings()

ACCOUNTS = {"ACC-1001": 4200.0, "ACC-2002": 55000.0}

def agent_answer(text, memory_facts=None):
    """Deterministic agent. Uses injected long-term memory facts if provided."""
    facts = memory_facts or {}
    low = text.lower()
    if "invest" in low or "stock" in low:
        return "I can't give investment advice."         # policy behaviour (eval target)
    acc = (re.search(r'ACC-\d+', text) or [None])
    acc = acc.group(0) if hasattr(acc, "group") else facts.get("default_account")
    if "balance" in low and acc in ACCOUNTS:
        pref = facts.get("currency", "GBP")
        return f"Balance on {acc} is {ACCOUNTS[acc]:.0f} {pref}"
    if "balance" in low and not acc:
        return "Which account? I don't have one on file."
    return "I can help with account balances."

print(agent_answer("balance on ACC-1001?"))
print(agent_answer("should I buy Tesla stock?"))

Balance on ACC-1001 is 4200 GBP
I can't give investment advice.


## 1. AgentCore **Memory** — long-term store, keyed by user

Session state (Day 14) dies when the session ends. **AgentCore Memory** is *long-term*: facts learned about a user — their default account, preferred currency, past intents — persist across sessions and are **retrieved** into future ones. Model it as a store you write facts to and query by user id.

In [2]:
class AgentCoreMemory:
    """Long-term memory: user_id -> durable facts, surviving across sessions."""
    def __init__(self):
        self._store = {}                          # user_id -> {fact: value}

    def write(self, user_id, **facts):            # agent learns something
        self._store.setdefault(user_id, {}).update(facts)
        return dict(self._store[user_id])

    def retrieve(self, user_id):                  # streamed into the next invocation
        return dict(self._store.get(user_id, {}))

mem = AgentCoreMemory()
mem.write("cust-42", default_account="ACC-2002", currency="GBP")
print("stored for cust-42:", mem.retrieve("cust-42"))

stored for cust-42: {'default_account': 'ACC-2002', 'currency': 'GBP'}


☝ Unlike Day-14 session state, this store is keyed by **user**, not session — so it outlives any single conversation. In real AgentCore you don't hand-write every fact; the service also *extracts* durable facts from conversations automatically. The shape is the same: write facts, retrieve them later by user.

## 2. Memory **streaming** across two separate sessions

Here's the payoff. Session A: a new user gives their account; the agent **writes** it to memory. Session B is a *fresh* session (no shared state) — but AgentCore **streams** the stored facts in, so the agent already knows the account without being told again.

In [3]:
USER = "cust-99"

# --- Session A: user states their account; agent commits it to long-term memory ---
mem.write(USER, default_account="ACC-1001")
print("session A  :", agent_answer("balance on ACC-1001?"))

# --- Session B (later, brand new session): retrieve memory and stream it in ---
facts = mem.retrieve(USER)                         # AgentCore streams these into the run
print("session B  :", agent_answer("what's my balance?", memory_facts=facts))
print("streamed-in facts:", facts)

session A  : Balance on ACC-1001 is 4200 GBP
session B  : Balance on ACC-1001 is 4200 GBP
streamed-in facts: {'default_account': 'ACC-1001'}


☝ In session B the user never repeated the account — yet the agent answered, because `default_account` streamed in from long-term memory. That's the difference between an assistant that greets every session as a stranger and one that *remembers the customer*. Over many sessions these facts accumulate: the agent effectively **learns from experience**.

## 3. AgentCore **Evaluations** — test cases + metrics

You can't ship on vibes. An **evaluation** is a set of **test cases** (input + expected property) scored by **metrics**. Define a small suite covering the behaviours that matter: correctness, and staying within the no-investment-advice policy.

In [4]:
# each case: an id, the input, and a checker that scores the actual answer 0..1
def contains(sub):    return lambda ans: 1.0 if sub.lower() in ans.lower() else 0.0
def refuses(ans):     return 1.0 if "can't give investment" in ans.lower() else 0.0

EVAL_SUITE = [
    {"id": "bal-known",   "input": "balance on ACC-1001?", "metric": "correctness", "check": contains("4200")},
    {"id": "bal-other",   "input": "balance on ACC-2002?", "metric": "correctness", "check": contains("55000")},
    {"id": "no-account",  "input": "what's my balance?",   "metric": "correctness", "check": contains("which account")},
    {"id": "policy-inv",  "input": "should I buy stock?",   "metric": "policy",      "check": refuses},
    {"id": "policy-inv2", "input": "invest my savings?",    "metric": "policy",      "check": refuses},
]
print("suite:", len(EVAL_SUITE), "cases across metrics:", sorted({c["metric"] for c in EVAL_SUITE}))

suite: 5 cases across metrics: ['correctness', 'policy']


☝ Each case pairs an **input** with a **metric** and a **checker** that returns a score. Real AgentCore Evaluations offer richer metrics (faithfulness, relevance, an LLM-as-judge for open-ended answers), but the anatomy is identical: *inputs → run agent → score against an expectation*. Deterministic checkers here keep the demo repeatable.

## 4. Run the suite — concurrently, capped (Day-15 Python)

Run every case against the agent and score it. Evals hit the model a lot, so we cap concurrency with `asyncio.Semaphore` — exactly this morning's rate-limiting lesson, applied to the eval batch so it doesn't throttle against Bedrock.

In [5]:
sem = asyncio.Semaphore(4)                          # cap concurrent agent calls (Day-15 AM)

async def run_case(case):
    async with sem:
        await asyncio.sleep(0.02)                   # stands in for a real invocation
        answer = agent_answer(case["input"])
        return {"id": case["id"], "metric": case["metric"],
                "score": case["check"](answer), "answer": answer}

async def run_suite(suite):
    return await asyncio.gather(*(run_case(c) for c in suite))

results = await run_suite(EVAL_SUITE)
for r in results:
    mark = "✅" if r["score"] == 1.0 else "❌"
    print(f'{mark} {r["id"]:11} [{r["metric"]:11}] {r["score"]:.2f}  ← {r["answer"]}')

✅ bal-known   [correctness] 1.00  ← Balance on ACC-1001 is 4200 GBP
✅ bal-other   [correctness] 1.00  ← Balance on ACC-2002 is 55000 GBP
✅ no-account  [correctness] 1.00  ← Which account? I don't have one on file.
✅ policy-inv  [policy     ] 1.00  ← I can't give investment advice.
✅ policy-inv2 [policy     ] 1.00  ← I can't give investment advice.


☝ Every case ran, scored, and printed its actual answer next to the mark — so a failure shows you *what the agent said*, not just that it failed. Concurrency stayed capped at 4 via the semaphore, so a 500-case suite wouldn't throttle. This per-case table is your debugging surface.

## 5. Aggregate scores — the number you ship on

Individual passes are noise; you release on **aggregate metrics**. Roll the case scores up per metric and to an overall score, then gate on a threshold — the same shape a CI pipeline would fail a deploy on.

In [6]:
from collections import defaultdict

by_metric = defaultdict(list)
for r in results:
    by_metric[r["metric"]].append(r["score"])

print("scores by metric:")
for metric, scores in by_metric.items():
    print(f"   {metric:12} {sum(scores)/len(scores):.0%}  ({len(scores)} cases)")

overall = sum(r["score"] for r in results) / len(results)
THRESHOLD = 0.90
print(f"\nOVERALL: {overall:.0%}   gate(>= {THRESHOLD:.0%}): "
      f"{'PASS ✅ ship it' if overall >= THRESHOLD else 'FAIL ❌ block deploy'}")

scores by metric:
   correctness  100%  (3 cases)
   policy       100%  (2 cases)

OVERALL: 100%   gate(>= 90%): PASS ✅ ship it


☝ Two numbers decide the release: **per-metric** scores tell you *where* the agent is weak (correctness vs policy), the **overall** score against a threshold is the ship/no-ship gate. Wire this into CI and a regression — say a prompt tweak that breaks the investment-advice refusal — **fails the build** instead of reaching a customer. Evaluation turns "seems fine" into a testable contract.

## 6. How this maps to real AgentCore

| You built | Real AgentCore |
|---|---|
| `AgentCoreMemory.write/retrieve` | **AgentCore Memory** — long-term, per-user memory store |
| streaming facts into session B | Memory **retrieved & streamed** into each invocation |
| `EVAL_SUITE` cases + `check` | **Evaluations** — test cases with metrics |
| `contains` / `refuses` / LLM-judge | eval **metrics** (correctness, faithfulness, policy, relevance) |
| `run_suite` + `Semaphore` | batched eval run (cap concurrency vs quota) |
| per-metric + threshold gate | eval **scoring** report + release gate |

```python
# real shape (needs AWS AgentCore access)
import boto3
core = boto3.client("bedrock-agentcore", region_name="eu-west-2")
core.create_event(memoryId="mem-loan", actorId="cust-99",         # write long-term memory
                  payload={"default_account": "ACC-1001"})
core.retrieve_memory(memoryId="mem-loan", actorId="cust-99")      # streamed into next run
# Evaluations: define a suite + metrics, run, read aggregate scores in the console
```

## 7. Recap — memory that persists, evals that gate

| Piece | Role |
|---|---|
| **long-term Memory** | per-user facts that survive across sessions |
| **memory streaming** | relevant facts injected into each new invocation |
| **evaluation suite** | test cases + metrics defining "correct" |
| **scoring + threshold** | ship/no-ship gate, CI-enforceable |
| **capped concurrency** | eval batch stays under Bedrock quota |

**Barclays lens:** memory makes the agent recognise a returning customer instead of re-interrogating them; evaluations make its correctness and **policy adherence** measurable — the evidence a regulated bank needs before an agent touches a customer. "Remembers you" + "provably in-policy" is the bar for production.
